Training the Classifier and Testing on Asian Tales. First, train a classifier on Western folktales that have ATU labels. Then, apply that classifier to Asian folktales and see how confident it is. The classifier should be less confident on Asian tales, showing that ATU categories don't work well for non-Western stories.

In [ ]:
# Setup
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import yaml
import logging
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import FolktaleDataLoader
from src.model import FolktaleClassifier
from src.visualization import (
    plot_confidence_distribution,
    plot_confidence_comparison
)

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

# Load configuration
with open('../config/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

print('Setup complete.')

Phase 1: Load and Prepare Training Data

In [ ]:
# Load data
loader = FolktaleDataLoader('../config/config.yaml')

try:
    western_df = loader.load_western_tales()
    print(f"Loaded {len(western_df)} Western folktales")
    
    # Preprocess texts
    western_df['text_processed'] = western_df['text'].apply(loader.preprocess_text)
    
    # Prepare training/validation split
    from sklearn.model_selection import train_test_split
    
    # Assuming 'atu_category' or similar label column
    if 'atu_category' not in western_df.columns:
        # Fill with placeholder if column doesn't exist
        print("Warning: 'atu_category' column not found. Using placeholder.")
        western_df['atu_category'] = [f'type_{i % 10}' for i in range(len(western_df))]
    
    texts = western_df['text_processed'].values
    labels = western_df['atu_category'].values
    
    # Split data
    test_size = config['training']['test_size']
    val_size = config['training']['validation_size']
    
    texts_train, texts_test, labels_train, labels_test = train_test_split(
        texts, labels, test_size=test_size, random_state=config['model']['random_state']
    )
    
    texts_train, texts_val, labels_train, labels_val = train_test_split(
        texts_train, labels_train, test_size=val_size, random_state=config['model']['random_state']
    )
    
    print(f"Training set: {len(texts_train)} tales")
    print(f"Validation set: {len(texts_val)} tales")
    print(f"Test set: {len(texts_test)} tales")
    
except FileNotFoundError as e:
    print(f"Error: {e}")
    print("Cannot proceed without Western folktales dataset.")

Train Classifier (Phase 1)

In [ ]:
# Initialize classifier
clf = FolktaleClassifier(config, device='cpu')  # Use 'cuda' if GPU available

# Train classifier
print("Training logistic regression classifier...")
metrics = clf.train(
    texts=texts_train,
    labels=labels_train,
    validation_texts=texts_val,
    validation_labels=labels_val
)

for key, value in metrics.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

# Save classifier
output_dir = Path('../results/models')
output_dir.mkdir(parents=True, exist_ok=True)
clf.save(str(output_dir / 'folktale_classifier.pkl'))
print(f"\n✓ Classifier saved to {output_dir / 'folktale_classifier.pkl'}")

Evaluate on Western Test Set

In [ ]:
# Test on Western folktales (held-out test set = our confidence baseline)
predictions_west, probabilities_west = clf.predict(texts_test)
confidence_metrics_west = clf.get_confidence_metrics(probabilities_west)
mahal_west = clf.mahalanobis_distances(texts_test)

n_classes = probabilities_west.shape[1]
print(f"Number of ATU classes: {n_classes}  (max entropy = log({n_classes}) = {np.log(n_classes):.3f} nats)")
print()
print("── Western test set ──────────────────────────────────")
print(f"  Max-softmax confidence : {confidence_metrics_west['mean_confidence']:.4f} ± {confidence_metrics_west['std_confidence']:.4f}")
print(f"  Margin (p1 - p2)       : {confidence_metrics_west['mean_margin']:.4f} ± {confidence_metrics_west['std_margin']:.4f}")
print(f"  Shannon entropy        : {confidence_metrics_west['mean_entropy']:.4f} ± {confidence_metrics_west['std_entropy']:.4f}")
print(f"  Normalized entropy H/log(K): {confidence_metrics_west['mean_normalized_entropy']:.4f}  (0=certain, 1=random guess)")
print(f"  Mahalanobis dist (min) : {mahal_west.mean():.4f} ± {mahal_west.std():.4f}")

plot_confidence_distribution(
    confidence_metrics_west['max_confidence'],
    confidence_metrics_west['entropy'],
    label='Western Folktales (Test Set)',
    output_path='../results/plots/02_western_confidence.png'
)

Phase 2: Test on Asian/SE Asian Folktales

In [ ]:
# Load Asian folktales and compute all confidence metrics
try:
    asian_df = loader.load_asian_tales()

    if len(asian_df) > 0:
        asian_df['text_processed'] = asian_df['text'].apply(loader.preprocess_text)
        texts_asian = asian_df['text_processed'].values
        print(f"Loaded {len(texts_asian)} Asian/SE Asian folktales")

        predictions_asian, probabilities_asian = clf.predict(texts_asian)
        confidence_metrics_asian = clf.get_confidence_metrics(probabilities_asian)
        mahal_asian = clf.mahalanobis_distances(texts_asian)

        print()
        print("── Asian/SE Asian folktales ──────────────────────────")
        print(f"  Max-softmax confidence : {confidence_metrics_asian['mean_confidence']:.4f} ± {confidence_metrics_asian['std_confidence']:.4f}")
        print(f"  Margin (p1 - p2)       : {confidence_metrics_asian['mean_margin']:.4f} ± {confidence_metrics_asian['std_margin']:.4f}")
        print(f"  Shannon entropy        : {confidence_metrics_asian['mean_entropy']:.4f} ± {confidence_metrics_asian['std_entropy']:.4f}")
        print(f"  Normalized entropy H/log(K): {confidence_metrics_asian['mean_normalized_entropy']:.4f}  (0=certain, 1=random guess)")
        print(f"  Mahalanobis dist (min) : {mahal_asian.mean():.4f} ± {mahal_asian.std():.4f}")

        plot_confidence_distribution(
            confidence_metrics_asian['max_confidence'],
            confidence_metrics_asian['entropy'],
            label='Asian/SE Asian Folktales',
            output_path='../results/plots/02_asian_confidence.png'
        )
    else:
        print("No Asian folktales loaded. Skipping Phase 2 testing.")
        confidence_metrics_asian = None
        mahal_asian = None

except Exception as e:
    print(f"Error loading Asian folktales: {e}")
    confidence_metrics_asian = None
    mahal_asian = None

Hypothesis Testing: Confidence Comparison

In [ ]:
if confidence_metrics_asian is not None:
    from scipy import stats
    import json

    def cohens_d(a, b):
        na, nb = len(a), len(b)
        pooled_std = np.sqrt(((na - 1) * np.std(a, ddof=1)**2 + (nb - 1) * np.std(b, ddof=1)**2) / (na + nb - 2))
        return (np.mean(a) - np.mean(b)) / pooled_std

    west_conf   = confidence_metrics_west['max_confidence']
    asian_conf  = confidence_metrics_asian['max_confidence']
    west_ne     = confidence_metrics_west['normalized_entropy']
    asian_ne    = confidence_metrics_asian['normalized_entropy']
    west_margin = confidence_metrics_west['margin']
    asian_margin= confidence_metrics_asian['margin']

    print("═══════════════════════════════════════════════════════")
    print("  Hypothesis: ATU classifier is less confident on Asian tales")
    print("═══════════════════════════════════════════════════════")

    metrics_table = [
        ("Max-softmax confidence", west_conf, asian_conf, "higher = more confident"),
        ("Normalized entropy H/log(K)", west_ne,  asian_ne,  "lower  = more confident; 1.0 = random guess"),
        ("Margin (p1 - p2)",            west_margin, asian_margin, "higher = more decisive"),
        ("Mahalanobis distance",        mahal_west, mahal_asian, "higher = further from training distribution"),
    ]

    print(f"\n{'Metric':<32} {'Western':>10} {'Asian':>10} {'Δ':>10} {'Cohen d':>9}  {'MW p-val':>10}")
    print("-" * 90)
    statistical_results = {}
    for label, w, a, note in metrics_table:
        delta = np.mean(w) - np.mean(a)
        d     = cohens_d(w, a)
        u_stat, p_val = stats.mannwhitneyu(w, a, alternative='two-sided')
        print(f"{label:<32} {np.mean(w):>10.4f} {np.mean(a):>10.4f} {delta:>+10.4f} {d:>9.3f}  {p_val:>10.2e}")
        print(f"  ({note})")
        statistical_results[label] = {'western_mean': float(np.mean(w)), 'asian_mean': float(np.mean(a)),
                                      'delta': float(delta), 'cohens_d': float(d), 'mw_p_value': float(p_val)}

    print()
    print("── Interpretation guide ──────────────────────────────")
    ne_asian = confidence_metrics_asian['mean_normalized_entropy']
    if ne_asian > 0.85:
        interp = "near-random (ATU does not generalize at all)"
    elif ne_asian > 0.65:
        interp = "highly uncertain (ATU generalizes poorly)"
    elif ne_asian > 0.40:
        interp = "moderately uncertain (partial generalization)"
    else:
        interp = "relatively confident (ATU may generalize)"
    print(f"  Asian normalized entropy {ne_asian:.3f} → {interp}")
    print(f"  Western normalized entropy {confidence_metrics_west['mean_normalized_entropy']:.3f} for comparison")
    print()

    # Check effect sizes (|d| > 0.5 = medium, > 0.8 = large)
    conf_d = cohens_d(west_conf, asian_conf)
    if abs(conf_d) >= 0.8:
        effect = "large"
    elif abs(conf_d) >= 0.5:
        effect = "medium"
    elif abs(conf_d) >= 0.2:
        effect = "small"
    else:
        effect = "negligible"
    print(f"  Confidence gap effect size: Cohen's d = {conf_d:.3f} ({effect})")

    plot_confidence_comparison(
        west_conf, asian_conf,
        output_path='../results/plots/02_confidence_comparison.png'
    )

    Path('../results/analysis').mkdir(parents=True, exist_ok=True)
    results = {
        'western_metrics': {k: v.tolist() if isinstance(v, np.ndarray) else v
                            for k, v in confidence_metrics_west.items()},
        'asian_metrics':   {k: v.tolist() if isinstance(v, np.ndarray) else v
                            for k, v in confidence_metrics_asian.items()},
        'mahalanobis': {'western_mean': float(mahal_west.mean()), 'asian_mean': float(mahal_asian.mean())},
        'statistical_tests': statistical_results,
    }
    with open('../results/analysis/02_confidence_analysis.json', 'w') as f:
        json.dump(results, f, indent=2)
    print("Results saved to ../results/analysis/02_confidence_analysis.json")
else:
    print("Cannot perform hypothesis test without Asian folktales data.")

Summary and Next Steps

In [ ]:
print(f"Trained logistic regression classifier on {len(texts_train)} Western folktales")
if confidence_metrics_asian is not None:
    print(f"Tested classifier on {len(texts_asian)} Asian/SE Asian folktales")
    print(f"Confidence difference: {west_conf - asian_conf:.4f} (Western > Asian)")
print("Next: Run notebook 03_interpretability.ipynb to:")
print("  1. Cluster Asian folktales using SAE-lens or k-means")
print("  2. Extract narrative motifs and archetypes")
print("  3. Analyze distinctive features per cluster")